# Purnazen skin-analysis model — end-to-end training

Trains the multi-head CNN that scores face scans and exports it as ONNX for the
FastAPI backend (`backend/app/ai/skin_model.py`). Runs top-to-bottom on **Google
Colab (GPU)** or a local machine with a CUDA GPU (CPU works too, just slower).

## Serving contract (keep in sync with `app/ai/skin_model.py`)
- **Input**: `1×3×224×224` float32, RGB, padded face crop, ImageNet-normalized
- **Output**: `1×8` float32 in `[0,1]` (sigmoid baked into the graph), scaled ×100 by the server
- **Head order** (`METRIC_ORDER`): hydration, oiliness, wrinkle, pigmentation, dark_circle, pore, elasticity, inflammation

## Which metrics are trained, and why
| Head | Supervision | Dataset |
|---|---|---|
| hydration_score | `dehydration` severity, inverted | killa92 |
| oiliness_score | `excessive oil` severity (+ blackheads proxy) | killa92 + GlowMix |
| wrinkle_score | `wrinkles/fine lines` severity | killa92 + GlowMix |
| pigmentation_score | `pigmentation/dark spots` severity | killa92 + GlowMix |
| dark_circle_score | `dark circles` severity | killa92 |
| pore_score | `open pores` severity | killa92 + GlowMix |
| elasticity_score | `skin elasticity` severity, inverted | killa92 |
| inflammation_score | `acne` severity (clinical proxy) | killa92 + GlowMix |

**Removed: `muscle_tone_score`.** No public dataset labels facial muscle tone, so it
cannot be trained or validated — it was dropped from the model heads and from the app
UI. The backend keeps its landmark-based CV estimate internally for the glow formula.

## Datasets
1. **[Facial Skin Analysis & Type Classification (killa92)](https://www.kaggle.com/datasets/killa92/facial-skin-analysis-and-type-classification)** — ~4k face images (640×640), 0–5 severity labels for ~18 conditions on a labeled subset. This is the *primary* (fully multi-label) supervision.
2. **[GlowMix merged facial skincare dataset](https://www.kaggle.com/datasets/drishyatomar/glowmix-merged-facial-skincare-dataset)** — ~10k images with per-condition class folders (acne, dark spots, pores, wrinkles, blackheads, clear). Used as *auxiliary weak supervision* (single-head labels, lower sample weight) so the scarce killa92 labels don't overfit.

Both are fuzzy-column-mapped in Step 3 — **read the printed mapping and fix
`MANUAL_COLUMN_OVERRIDES` / `AUX_CLASS_RULES` if anything resolved wrong.**

## Step 0 — Environment
Installs everything missing. On Colab, `torch`/`torchvision` are preinstalled with CUDA.

In [ ]:
%pip install -q torch torchvision pandas openpyxl opencv-python-headless mediapipe onnx onnxruntime scipy matplotlib tqdm kaggle

import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

In [ ]:
from pathlib import Path
import os, random, numpy as np, torch

# ── Contract (MUST match backend/app/ai/skin_model.py) ─────────────────────
METRIC_ORDER = (
    'hydration_score',
    'oiliness_score',
    'wrinkle_score',
    'pigmentation_score',
    'dark_circle_score',
    'pore_score',
    'elasticity_score',
    'inflammation_score',
)
N_HEADS = len(METRIC_ORDER)
INPUT_SIZE = 224
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
SEVERITY_MAX = 5.0   # dataset labels are 0..5

# ── Paths ───────────────────────────────────────────────────────────────────
ROOT = Path('data')
RAW_KILLA = ROOT / 'raw' / 'killa92'
RAW_GLOWMIX = ROOT / 'raw' / 'glowmix'
CROPS_DIR = ROOT / 'crops'
CKPT_DIR = Path('checkpoints')
for p in (RAW_KILLA, RAW_GLOWMIX, CROPS_DIR, CKPT_DIR):
    p.mkdir(parents=True, exist_ok=True)

# If this notebook lives in backend/ml of the repo, export straight into the backend.
_repo_target = Path('..') / 'app' / 'ai' / 'models'
EXPORT_PATH = (_repo_target / 'skin_model.onnx') if _repo_target.parent.exists() else Path('skin_model.onnx')

# ── Hyperparameters ─────────────────────────────────────────────────────────
CFG = dict(
    seed=42,
    backbone='efficientnet_b0',
    batch_size=32,
    head_epochs=3,        # phase 1: frozen backbone, train the head
    epochs=35,            # phase 2: full fine-tune
    lr_head=2e-3,
    lr_finetune=3e-4,
    weight_decay=1e-4,
    warmup_epochs=3,
    ema_decay=0.999,
    early_stop_patience=8,
    aux_sample_weight=0.35,   # GlowMix weak labels count less than killa92 severity labels
    aux_cap_per_class=1500,   # balance the auxiliary classes
    use_glowmix=True,
)

SMOKE = False  # True = 200 images / 2 epochs, just to prove the pipeline end-to-end

random.seed(CFG['seed']); np.random.seed(CFG['seed']); torch.manual_seed(CFG['seed'])
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '| export target:', EXPORT_PATH)

## Step 1 — Download the datasets (Kaggle API)

Needs a Kaggle API token: kaggle.com → *Settings → API → Create New Token* →
place the downloaded `kaggle.json` at `~/.kaggle/kaggle.json` (the cell below
offers an upload prompt on Colab). Re-running skips anything already downloaded.

In [ ]:
import shutil, subprocess, sys

kaggle_json = Path.home() / '.kaggle' / 'kaggle.json'
if not kaggle_json.exists():
    try:  # Colab: prompt an upload
        from google.colab import files  # type: ignore
        print('Upload your kaggle.json:')
        up = files.upload()
        kaggle_json.parent.mkdir(parents=True, exist_ok=True)
        kaggle_json.write_bytes(up['kaggle.json'])
        kaggle_json.chmod(0o600)
    except ImportError:
        raise SystemExit('Place your Kaggle API token at ~/.kaggle/kaggle.json first.')

def kaggle_download(slug: str, dest: Path):
    if any(dest.rglob('*.jpg')) or any(dest.rglob('*.png')) or any(dest.rglob('*.jpeg')):
        print(f'✓ {slug} already present in {dest}')
        return
    subprocess.run([sys.executable, '-m', 'kaggle', 'datasets', 'download', '-d', slug,
                    '-p', str(dest), '--unzip'], check=True)
    print(f'✓ downloaded {slug} → {dest}')

kaggle_download('killa92/facial-skin-analysis-and-type-classification', RAW_KILLA)
if CFG['use_glowmix']:
    kaggle_download('drishyatomar/glowmix-merged-facial-skincare-dataset', RAW_GLOWMIX)

## Step 2 — Discover what we actually downloaded
Kaggle datasets shuffle their internal layout between versions, so never hard-code
paths — index every image by basename and locate the Excel/CSV label files.

In [ ]:
import pandas as pd
from collections import Counter

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def index_images(root: Path) -> dict:
    """basename (lowercase) -> full path, across any nesting."""
    idx = {}
    for p in root.rglob('*'):
        if p.suffix.lower() in IMG_EXTS:
            idx[p.name.lower()] = p
    return idx

killa_images = index_images(RAW_KILLA)
label_files = sorted(list(RAW_KILLA.rglob('*.xlsx')) + list(RAW_KILLA.rglob('*.csv')))
print(f'killa92: {len(killa_images)} images | label files: {[f.name for f in label_files]}')

label_frames = []
for f in label_files:
    df = pd.read_excel(f) if f.suffix == '.xlsx' else pd.read_csv(f)
    df['__source__'] = f.name
    label_frames.append(df)
    print(f'\n── {f.name} ── {len(df)} rows, columns:')
    print(list(df.columns))

raw_labels = pd.concat(label_frames, ignore_index=True) if label_frames else pd.DataFrame()
assert not raw_labels.empty, 'No label files found under ' + str(RAW_KILLA)
raw_labels.head()

## Step 3 — Map label columns → metric heads
Fuzzy keyword matching resolves each spreadsheet column to a head. `invert=True`
flips severity for the two *good-is-high* heads: `dehydration 5` → `hydration 0`,
and elasticity-loss severity → elasticity score.

**Read the printed mapping.** If a column resolved wrongly (or the sheet uses a
name the keywords miss), pin it in `MANUAL_COLUMN_OVERRIDES`.

In [ ]:
import re

def norm(s: str) -> str:
    return re.sub(r'[^a-z0-9]+', ' ', str(s).lower()).strip()

# keyword sets per head — matched against normalized column names
COLUMN_HINTS = {
    'hydration_score':    (['dehydration', 'dryness', 'dry skin'], True),
    'oiliness_score':     (['excessive oil', 'oily', 'oil', 'sebum'], False),
    'wrinkle_score':      (['wrinkle', 'fine line'], False),
    'pigmentation_score': (['pigmentation', 'dark spot', 'uneven skin tone', 'freckle'], False),
    'dark_circle_score':  (['dark circle', 'under eye', 'eye puffiness', 'eye bag'], False),
    'pore_score':         (['open pore', 'pore'], False),
    'elasticity_score':   (['elasticity', 'sagging', 'firmness'], True),
    'inflammation_score': (['acne', 'redness', 'breakout', 'pimple', 'inflammation'], False),
}

# column-name → head, pinned by hand after reading the printout below.
MANUAL_COLUMN_OVERRIDES: dict[str, str] = {
    # 'Wrinkles on forehead': 'wrinkle_score',
}

# find the image-filename column
file_col = next((c for c in raw_labels.columns
                 if norm(c) in ('file name', 'filename', 'image', 'image name', 'img', 'file')), None)
assert file_col, f'Could not find a filename column in {list(raw_labels.columns)}'

resolved: dict[str, tuple[str, bool]] = {}   # column -> (head, invert)
for col in raw_labels.columns:
    if col in (file_col, '__source__'):
        continue
    if col in MANUAL_COLUMN_OVERRIDES:
        head = MANUAL_COLUMN_OVERRIDES[col]
        resolved[col] = (head, COLUMN_HINTS[head][1])
        continue
    n = norm(col)
    for head, (keys, invert) in COLUMN_HINTS.items():
        if any(k in n for k in keys):
            # first match wins per column; multiple columns may map to one head
            resolved[col] = (head, invert)
            break

print('Resolved column mapping (column → head, invert):')
for col, (head, inv) in resolved.items():
    print(f'  {col!r:40s} → {head:22s} invert={inv}')
unmapped = [c for c in raw_labels.columns if c not in resolved and c not in (file_col, '__source__')]
print('\nUnmapped columns (ignored):', unmapped)

mapped_heads = {h for h, _ in resolved.values()}
assert len(mapped_heads) >= 6, f'Only {len(mapped_heads)} heads mapped — fix MANUAL_COLUMN_OVERRIDES'

# Build the primary labeled dataframe: 0–5 severity → 0–100 target + mask per head.
rows = []
for _, r in raw_labels.iterrows():
    img_path = killa_images.get(str(r[file_col]).lower())
    if img_path is None:
        continue
    row = {'filepath': str(img_path), 'weight': 1.0}
    per_head_vals: dict[str, list[float]] = {}
    for col, (head, invert) in resolved.items():
        v = pd.to_numeric(r.get(col), errors='coerce')
        if pd.isna(v):
            continue
        frac = min(max(float(v) / SEVERITY_MAX, 0.0), 1.0)
        per_head_vals.setdefault(head, []).append((1.0 - frac if invert else frac) * 100.0)
    for head in METRIC_ORDER:
        vals = per_head_vals.get(head)
        row[head] = float(np.mean(vals)) if vals else 50.0
        row[head + '_mask'] = 1.0 if vals else 0.0
    if any(row[h + '_mask'] for h in METRIC_ORDER):
        rows.append(row)

primary = pd.DataFrame(rows).drop_duplicates(subset='filepath').reset_index(drop=True)
print(f'\nPrimary labeled rows: {len(primary)}')
print('Label coverage per head:')
for h in METRIC_ORDER:
    print(f'  {h:22s} {int(primary[h + "_mask"].sum()):5d} labeled')

## Step 4 — GlowMix auxiliary weak labels
GlowMix is organised as class folders (one condition per image). Each image gets a
*single-head* label (severity proxy) with every other head masked out, and a lower
sample weight so weak labels can't overwhelm the real severity labels.
`clear/normal` folders label all mapped condition heads as low.

In [ ]:
# folder-name keywords → (head, proxy score 0-100). None = 'clear skin' rule.
AUX_CLASS_RULES = [
    (['acne', 'pimple', 'breakout'],        ('inflammation_score', 75.0)),
    (['dark spot', 'darkspot', 'pigment'],  ('pigmentation_score', 75.0)),
    (['pore'],                              ('pore_score', 75.0)),
    (['wrinkle', 'fine line'],              ('wrinkle_score', 75.0)),
    (['blackhead'],                         ('oiliness_score', 65.0)),  # oily-skin proxy
    (['clear', 'normal', 'healthy'],        None),
]
CLEAR_HEADS = ['inflammation_score', 'pigmentation_score', 'pore_score', 'wrinkle_score', 'oiliness_score']

aux_rows = []
if CFG['use_glowmix'] and RAW_GLOWMIX.exists():
    class_dirs = sorted({p.parent for p in RAW_GLOWMIX.rglob('*') if p.suffix.lower() in IMG_EXTS})
    print('GlowMix class folders found:')
    rng = np.random.default_rng(CFG['seed'])
    for d in class_dirs:
        n = norm(d.name)
        rule = next((r for keys, r in AUX_CLASS_RULES if any(k in n for k in keys)), 'SKIP')
        imgs = [p for p in d.iterdir() if p.suffix.lower() in IMG_EXTS]
        print(f'  {d.name!r:30s} {len(imgs):6d} imgs → {rule if rule != "SKIP" else "(no rule — skipped)"}')
        if rule == 'SKIP':
            continue
        if len(imgs) > CFG['aux_cap_per_class']:
            imgs = list(rng.choice(imgs, CFG['aux_cap_per_class'], replace=False))
        for p in imgs:
            row = {'filepath': str(p), 'weight': CFG['aux_sample_weight']}
            for h in METRIC_ORDER:
                row[h], row[h + '_mask'] = 50.0, 0.0
            if rule is None:  # clear skin → low severity on all condition heads
                for h in CLEAR_HEADS:
                    row[h], row[h + '_mask'] = 15.0, 1.0
            else:
                head, score = rule
                row[head], row[head + '_mask'] = score, 1.0
            aux_rows.append(row)

aux = pd.DataFrame(aux_rows)
print(f'\nAuxiliary rows: {len(aux)}')

## Step 5 — Face crops (cached)
Serving crops a padded face box from MediaPipe landmarks before resizing to 224.
Training must match: MediaPipe face detection → padded crop (fallback: centred
crop, same as the server's `_face_crop` fallback). Crops are cached to disk so
re-runs are instant.

In [ ]:
import cv2, hashlib
from tqdm.auto import tqdm
import mediapipe as mp

mp_fd = mp.solutions.face_detection

def crop_face(img_bgr, detector):
    h, w = img_bgr.shape[:2]
    res = detector.process(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    if res.detections:
        box = max(res.detections, key=lambda d: d.score[0]).location_data.relative_bounding_box
        # padding mirrors app/ai/skin_model._face_crop margins
        x1 = max(0, int((box.xmin - 0.06) * w)); x2 = min(w, int((box.xmin + box.width + 0.06) * w))
        y1 = max(0, int((box.ymin - 0.10) * h)); y2 = min(h, int((box.ymin + box.height + 0.06) * h))
        if x2 - x1 >= 20 and y2 - y1 >= 20:
            return img_bgr[y1:y2, x1:x2]
    bw, bh = int(w * 0.62), int(h * 0.64)   # server fallback: centred crop
    return img_bgr[int(h * 0.16):int(h * 0.16) + bh, (w - bw) // 2:(w - bw) // 2 + bw]

all_rows = pd.concat([primary, aux], ignore_index=True) if len(aux) else primary.copy()
if SMOKE:
    all_rows = all_rows.sample(min(200, len(all_rows)), random_state=CFG['seed']).reset_index(drop=True)

crop_paths, kept = [], []
with mp_fd.FaceDetection(model_selection=0, min_detection_confidence=0.4) as det:
    for i, fp in enumerate(tqdm(all_rows['filepath'], desc='cropping')):
        out = CROPS_DIR / (Path(fp).stem + '_' + hashlib.md5(fp.encode()).hexdigest()[:8] + '.jpg')
        if not out.exists():
            img = cv2.imread(fp)
            if img is None:
                continue
            crop = crop_face(img, det)
            if crop.size == 0:
                continue
            cv2.imwrite(str(out), crop, [cv2.IMWRITE_JPEG_QUALITY, 95])
        crop_paths.append(str(out)); kept.append(i)

data = all_rows.iloc[kept].reset_index(drop=True)
data['croppath'] = crop_paths
print(f'Usable rows after cropping: {len(data)}')

In [ ]:
# Split: only *fully supervised* killa92 rows are eligible for val/test
# (weak GlowMix labels would make the metrics meaningless). Aux → train only.
is_primary = data['weight'] == 1.0
prim_idx = data.index[is_primary].to_numpy()
rng = np.random.default_rng(CFG['seed'])
rng.shuffle(prim_idx)
n_test = max(1, int(len(prim_idx) * 0.15))
n_val = max(1, int(len(prim_idx) * 0.15))
test_idx = set(prim_idx[:n_test]); val_idx = set(prim_idx[n_test:n_test + n_val])

data['split'] = ['test' if i in test_idx else 'val' if i in val_idx else 'train' for i in data.index]
print(data['split'].value_counts().to_string())
print(f"(train includes {int(((data['split'] == 'train') & ~is_primary).sum())} weak GlowMix rows)")

## Step 6 — Dataset, augmentation, model
- Augmentation is deliberately **colour-gentle**: hue/saturation shifts would corrupt
  the very signals (redness, pigmentation, dark circles) the model must learn.
- Backbone: **EfficientNet-B0** (ImageNet pretrained) with an 8-unit regression head.
  ~5.3M params — comfortably real-time on the backend's CPU via ONNX Runtime.
- Loss: **masked, sample-weighted SmoothL1** on sigmoid outputs; per-head weights
  balance scarce heads against abundant ones.

In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(INPUT_SIZE, scale=(0.75, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.08, hue=0.01),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.10, scale=(0.02, 0.08)),
])
eval_tf = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),  # matches serving preprocess exactly
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class SkinDataset(Dataset):
    def __init__(self, df, tf):
        self.df, self.tf = df.reset_index(drop=True), tf
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        x = self.tf(Image.open(r['croppath']).convert('RGB'))
        y = torch.tensor([r[h] / 100.0 for h in METRIC_ORDER], dtype=torch.float32)
        m = torch.tensor([r[h + '_mask'] for h in METRIC_ORDER], dtype=torch.float32)
        w = torch.tensor(float(r['weight']), dtype=torch.float32)
        return x, y, m, w

df_train = data[data['split'] == 'train']
df_val = data[data['split'] == 'val']
df_test = data[data['split'] == 'test']

# sampler: draw primary rows ~3× more often than weak aux rows
sample_w = np.where(df_train['weight'].to_numpy() == 1.0, 3.0, 1.0)
sampler = WeightedRandomSampler(torch.as_tensor(sample_w, dtype=torch.double), len(df_train), replacement=True)

NUM_WORKERS = 2 if os.name != 'nt' else 0
train_loader = DataLoader(SkinDataset(df_train, train_tf), batch_size=CFG['batch_size'],
                          sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(SkinDataset(df_val, eval_tf), batch_size=CFG['batch_size'], num_workers=NUM_WORKERS)
test_loader = DataLoader(SkinDataset(df_test, eval_tf), batch_size=CFG['batch_size'], num_workers=NUM_WORKERS)

# per-head loss weights ∝ 1/sqrt(label count) — scarce heads still get gradient
counts = np.array([max(1.0, df_train[h + '_mask'].sum()) for h in METRIC_ORDER])
head_w = np.sqrt(counts.max() / counts)
HEAD_WEIGHTS = torch.tensor(head_w / head_w.mean(), dtype=torch.float32, device=DEVICE)
print('Per-head loss weights:', dict(zip(METRIC_ORDER, np.round(head_w / head_w.mean(), 2))))

In [ ]:
import copy
import torch.nn as nn
from torchvision import models

def build_model():
    net = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    in_f = net.classifier[-1].in_features
    net.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_f, N_HEADS))
    return net

class Ema:
    """Exponential moving average of weights — the exported model uses these."""
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = copy.deepcopy(model).eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        for s, p in zip(self.shadow.state_dict().values(), model.state_dict().values()):
            if s.dtype.is_floating_point:
                s.mul_(self.decay).add_(p.detach(), alpha=1 - self.decay)
            else:
                s.copy_(p)

smooth_l1 = nn.SmoothL1Loss(beta=0.1, reduction='none')

def masked_loss(logits, target, mask, sample_w):
    per = smooth_l1(torch.sigmoid(logits), target)          # B×H
    per = per * mask * HEAD_WEIGHTS * sample_w[:, None]
    return per.sum() / (mask * sample_w[:, None]).sum().clamp(min=1.0)

@torch.no_grad()
def evaluate(model, loader):
    """Returns (masked MAE overall 0-100, per-head MAE list)."""
    model.eval()
    abs_err = torch.zeros(N_HEADS); cnt = torch.zeros(N_HEADS)
    for x, y, m, _ in loader:
        p = torch.sigmoid(model(x.to(DEVICE))).cpu()
        abs_err += ((p - y).abs() * m).sum(0) * 100.0
        cnt += m.sum(0)
    per_head = (abs_err / cnt.clamp(min=1)).tolist()
    overall = float(abs_err.sum() / cnt.sum().clamp(min=1))
    return overall, per_head

model = build_model().to(DEVICE)
print(sum(p.numel() for p in model.parameters()) / 1e6, 'M params')

## Step 7 — Train
Phase 1 trains only the new head on a frozen backbone (stabilises the pretrained
features), then phase 2 fine-tunes everything with warmup → cosine decay, AMP,
gradient clipping and EMA. Best checkpoint = lowest **EMA val MAE**.

In [ ]:
import math, time

EPOCHS = 2 if SMOKE else CFG['epochs']
HEAD_EPOCHS = 1 if SMOKE else CFG['head_epochs']
scaler = torch.amp.GradScaler(enabled=(DEVICE == 'cuda'))

def run_train_epoch(model, opt, lr):
    for g in opt.param_groups:
        g['lr'] = lr
    model.train()
    tot, n = 0.0, 0
    for x, y, m, w in train_loader:
        x, y, m, w = x.to(DEVICE), y.to(DEVICE), m.to(DEVICE), w.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE, enabled=(DEVICE == 'cuda')):
            loss = masked_loss(model(x), y, m, w)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        ema.update(model)
        tot += float(loss) * x.size(0); n += x.size(0)
    return tot / max(1, n)

# ── Phase 1: head only ──────────────────────────────────────────────────────
for p in model.features.parameters():
    p.requires_grad_(False)
ema = Ema(model, CFG['ema_decay'])
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                        lr=CFG['lr_head'], weight_decay=CFG['weight_decay'])
for ep in range(1, HEAD_EPOCHS + 1):
    tl = run_train_epoch(model, opt, CFG['lr_head'])
    vm, _ = evaluate(model, val_loader)
    print(f'[head {ep}/{HEAD_EPOCHS}] train_loss={tl:.4f}  val_MAE={vm:.2f}')

# ── Phase 2: full fine-tune ─────────────────────────────────────────────────
for p in model.parameters():
    p.requires_grad_(True)
opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr_finetune'], weight_decay=CFG['weight_decay'])

def lr_at(ep):  # linear warmup → cosine
    if ep <= CFG['warmup_epochs']:
        return CFG['lr_finetune'] * ep / CFG['warmup_epochs']
    t = (ep - CFG['warmup_epochs']) / max(1, EPOCHS - CFG['warmup_epochs'])
    return CFG['lr_finetune'] * 0.5 * (1 + math.cos(math.pi * t))

best_mae, best_ep, since_best = float('inf'), 0, 0
CKPT = CKPT_DIR / 'best_nb.pt'
for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    tl = run_train_epoch(model, opt, lr_at(ep))
    vm, per = evaluate(ema.shadow.to(DEVICE), val_loader)
    mae_str = ' '.join(f'{h.split("_")[0][:5]}:{e:.1f}' for h, e in zip(METRIC_ORDER, per))
    star = ''
    if vm < best_mae:
        best_mae, best_ep, since_best, star = vm, ep, 0, '  ★ saved'
        torch.save({'state_dict': ema.shadow.state_dict(), 'metric_order': METRIC_ORDER,
                    'backbone': CFG['backbone'], 'val_mae': vm}, CKPT)
    else:
        since_best += 1
    print(f'ep {ep:3d}/{EPOCHS} lr={lr_at(ep):.2e} loss={tl:.4f} valMAE={vm:5.2f} [{mae_str}]{star}')
    if since_best >= CFG['early_stop_patience']:
        print(f'Early stop — no val improvement for {since_best} epochs.')
        break
print(f'\nBest val MAE {best_mae:.2f} at epoch {best_ep} → {CKPT}')

## Step 8 — Evaluate on the held-out test set
Per-head MAE (0–100), Pearson *r* and Spearman *ρ* against the severity labels,
with horizontal-flip test-time augmentation (matches nothing at serve time, but
flip-invariance is a fair accuracy boost we can also add server-side later).

**Export gate:** only ship the model if it clearly beats the classical-CV baseline
(run `backend/sandbox/test_analyzers.py` on the same crops for comparison).

In [ ]:
from scipy import stats

best = build_model().to(DEVICE)
best.load_state_dict(torch.load(CKPT, map_location=DEVICE, weights_only=False)['state_dict'])
best.eval()

preds, targs, masks = [], [], []
with torch.no_grad():
    for x, y, m, _ in test_loader:
        x = x.to(DEVICE)
        p = (torch.sigmoid(best(x)) + torch.sigmoid(best(torch.flip(x, dims=[3])))) / 2  # hflip TTA
        preds.append(p.cpu()); targs.append(y); masks.append(m)
preds = torch.cat(preds).numpy() * 100
targs = torch.cat(targs).numpy() * 100
masks = torch.cat(masks).numpy().astype(bool)

report = []
for i, h in enumerate(METRIC_ORDER):
    sel = masks[:, i]
    if sel.sum() < 3:
        report.append({'head': h, 'n': int(sel.sum()), 'MAE': np.nan, 'pearson_r': np.nan, 'spearman_rho': np.nan})
        continue
    p, t = preds[sel, i], targs[sel, i]
    report.append({
        'head': h, 'n': int(sel.sum()),
        'MAE': float(np.abs(p - t).mean()),
        'pearson_r': float(stats.pearsonr(p, t)[0]) if np.std(t) > 0 else np.nan,
        'spearman_rho': float(stats.spearmanr(p, t)[0]) if np.std(t) > 0 else np.nan,
    })
report = pd.DataFrame(report).set_index('head').round(2)
print(report.to_string())

In [ ]:
import matplotlib.pyplot as plt

INK, MUTED, BLUE = '#374151', '#9ca3af', '#4269d0'

# Test MAE per head — one measure, one hue; grid recessive; direct value labels.
r = report.dropna(subset=['MAE']).sort_values('MAE')
fig, ax = plt.subplots(figsize=(7, 3.6))
bars = ax.barh([h.replace('_score', '') for h in r.index], r['MAE'], color=BLUE, height=0.62)
for b, v in zip(bars, r['MAE']):
    ax.text(b.get_width() + 0.3, b.get_y() + b.get_height() / 2, f'{v:.1f}', va='center', fontsize=9, color=INK)
ax.set_xlabel('Test MAE (0–100 scale, lower is better)', color=INK)
ax.set_title('Per-head test error', loc='left', color=INK, fontsize=11)
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(colors=INK)
ax.xaxis.grid(True, color='#e5e7eb', linewidth=0.8); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

# Predicted vs. label per head — y=x reference, single hue, one axis pair each.
heads_ok = [h for h in METRIC_ORDER if masks[:, METRIC_ORDER.index(h)].sum() >= 3]
ncols = 4; nrows = math.ceil(len(heads_ok) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(3.1 * ncols, 3.0 * nrows))
for ax, h in zip(np.ravel(axes), heads_ok):
    i = METRIC_ORDER.index(h); sel = masks[:, i]
    ax.plot([0, 100], [0, 100], ls='--', lw=1, color=MUTED)
    ax.scatter(targs[sel, i], preds[sel, i], s=14, alpha=0.55, color=BLUE, edgecolors='none')
    ax.set_title(h.replace('_score', ''), fontsize=10, color=INK)
    ax.set_xlim(0, 100); ax.set_ylim(0, 100)
    ax.spines[['top', 'right']].set_visible(False); ax.tick_params(labelsize=8, colors=INK)
for ax in np.ravel(axes)[len(heads_ok):]:
    ax.axis('off')
fig.supxlabel('label', fontsize=10, color=INK); fig.supylabel('prediction', fontsize=10, color=INK)
plt.tight_layout(); plt.show()

## Step 9 — Export ONNX for the backend
Bakes the final `sigmoid` into the graph (the server does **not** re-apply it),
verifies torch↔onnxruntime parity, and checks the `1×8` contract.

In [ ]:
import onnxruntime as ort

class WithSigmoid(nn.Module):
    def __init__(self, net):
        super().__init__(); self.net = net
    def forward(self, x):
        return torch.sigmoid(self.net(x))

export_model = WithSigmoid(best).eval().cpu()
dummy = torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE)
EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.onnx.export(
    export_model, dummy, str(EXPORT_PATH),
    input_names=['input'], output_names=['scores'],
    opset_version=17, do_constant_folding=True, dynamo=False,
)

sess = ort.InferenceSession(str(EXPORT_PATH), providers=['CPUExecutionProvider'])
x = torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE)
with torch.no_grad():
    ref = export_model(x).numpy()
out = sess.run(None, {'input': x.numpy()})[0]
assert out.shape == (1, N_HEADS), f'Contract violation: output {out.shape} != (1, {N_HEADS})'
diff = float(np.abs(ref - out).max())
assert diff < 1e-4, f'torch↔onnx mismatch: {diff}'
print(f'✓ exported {EXPORT_PATH} | output shape {out.shape} | max parity diff {diff:.2e}')

try:  # Colab: offer the file as a download
    from google.colab import files  # type: ignore
    files.download(str(EXPORT_PATH))
except ImportError:
    pass

## Step 10 — Deploy
1. Drop the file at **`backend/app/ai/models/skin_model.onnx`** (already done if you
   ran this notebook inside `backend/ml/` — see the export path above).
2. Restart the backend. `app/ai/skin_model.py` picks it up automatically and the
   scan pipeline switches `scoring_method` from `"cv"` to `"model"` (visible in each
   scan's `raw_metrics`).
3. Safety: if the file is missing/corrupt or its head count doesn't match
   `METRIC_ORDER`, the server logs a warning and falls back to the CV analyzers —
   a bad export can't break scans.

**Contract reminders**
- The 8-head `METRIC_ORDER` here must equal `backend/app/ai/skin_model.py::METRIC_ORDER`
  and `backend/ml/common.py::METRIC_ORDER`.
- `muscle_tone_score` is intentionally **not** a model head (no dataset). The backend
  computes it with the landmark CV analyzer; it is no longer shown in the app.
- Only export after the test-set MAE/correlations clearly beat the CV baseline
  (`backend/sandbox/test_analyzers.py` on the same crops).